# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

# ============================================================================
# FRESH START CONTROL - Set to True to wipe all checkpoints and start over
# ============================================================================
FRESH_START = True   # <-- Change to True to clear ALL saved progress
# ============================================================================

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print(f"FRESH_START = {FRESH_START}")
print("=" * 60)

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-03-26
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Breast_cancer_data.csv",  # Path to your CSV file
    "target_column": "diagnosis",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Breast Cancer Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    # All 5 features are continuous; no categorical columns needed:
    "categorical_columns": [],
    "task_type": "classification",

    # ========== OPTIONAL: Fairness Evaluation ==========
    # Protected attribute for fairness metrics (use cleaned column name after preprocessing).
    # This dataset has no demographic columns, so fairness evaluation is not applicable.
    "protected_col": None,

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # 569 rows - small enough to use all
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # No missing values in this dataset
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "pategan", "medgan"],                       # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "pategan", "medgan"],  

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "full",                       # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 15,                        # Trials for smoke testing / pilot phase
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Protected column: {NOTEBOOK_CONFIG.get('protected_col', None)}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Breast_cancer_data.csv
  Target column: diagnosis
  Protected column: None
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  Tuning mode: full


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

if not FRESH_START and 'checkpoint' in dir() and hasattr(checkpoint, 'exists') and checkpoint.exists('section_2'):
    saved = checkpoint.load('section_2')
    data = saved['data']
    original_data = saved['original_data']
    target_column = saved['target_column']
    DATASET_IDENTIFIER = saved['DATASET_IDENTIFIER']
    categorical_columns = saved['categorical_columns']
    metadata = saved['metadata']
    models_to_run = saved['models_to_run']
    RUN_MODE = saved['RUN_MODE']
    TARGET_COLUMN = target_column
    print("[RESUME] Loaded Section 2 from checkpoint")
else:
    # Load and preprocess data using the config
    (
        data,                  # Processed DataFrame
        original_data,         # Copy for reference
        target_column,         # Target column name (cleaned)
        DATASET_IDENTIFIER,    # Dataset identifier for results paths
        categorical_columns,   # List of categorical columns
        metadata               # Full preprocessing metadata
    ) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

    # Set aliases for backward compatibility with existing notebook cells
    TARGET_COLUMN = target_column

    # Get models to run based on config
    models_to_run = get_models_to_run(NOTEBOOK_CONFIG)

    # Set RUN_MODE for backward compatibility with Section 4 tuning cells
    RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

# Initialize checkpoint system (now that DATASET_IDENTIFIER is known)
checkpoint = SectionCheckpoint(DATASET_IDENTIFIER)

# If FRESH_START, wipe all checkpoints and optimization studies
if FRESH_START:
    n_removed = checkpoint.clear_all()
    print(f"[FRESH START] Cleared {n_removed} checkpoint(s) and optimization studies")

existing = checkpoint.list_checkpoints()
if existing:
    print(f"[CHECKPOINT] Found {len(existing)} existing checkpoint(s): {existing}")

# Save Section 2 checkpoint (idempotent - overwrites if exists)
if not checkpoint.exists('section_2'):
    checkpoint.save('section_2', {
        'data': data, 'original_data': original_data,
        'target_column': target_column, 'DATASET_IDENTIFIER': DATASET_IDENTIFIER,
        'categorical_columns': categorical_columns, 'metadata': metadata,
        'models_to_run': models_to_run, 'RUN_MODE': RUN_MODE,
    })

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Breast_cancer_data.csv
[LOAD] Loaded 569 rows, 6 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (569, 6)
[PREPROCESS] Categorical columns: []
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (569, 6)
[PREPROCESS] Dataset identifier: breast-cancer-data
[FLUSH] No previous studies found in results/breast-cancer-data/optimization_studies — starting clean
[FRESH START] Cleared 0 checkpoint(s) and optimization studies

PREPROCESSING COMPLETE
  Dataset identifier: breast-cancer-data
  Processed shape: (569, 6)
  Target column: diagnosis
  Task type: classification
  Categorical columns: []
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: full
  Session: 2026-03-26
  Results path: results/breast-cancer-data/2026-03-26/Section-2


### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Breast Cancer Dataset
Target column: diagnosis
Results path: results/breast-cancer-data/2026-03-26/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Breast Cancer Dataset
   Shape......................... 569 rows x 6 columns
   Memory Usage.................. 0.03 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 0
   Numeric Columns............... 6
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (6 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.594 (Moderately Imbalanced)

[4/6] Feature Distributions...
   Saved: 1 distribution file(s) (5 features)

[5/6] Correlation Analysis...
   Saved: mixed_association_heatmap.png
   Saved: association_matrix.csv
   Saved: target_correlations.csv (5 features)

[6/6] Configuration Validation

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models (checkpoint resumes completed models)
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True,
    checkpoint=checkpoint
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success' and result.get('model') is not None:
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (569, 6)
Target column: diagnosis
Samples to generate: 569
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda


[1/7] Training CTGAN...
--------------------------------------------------
  Device: cuda
  Training CTGAN...


Gen. (-0.48) | Discrim. (-0.07): 100%|██████████| 300/300 [00:04<00:00, 65.66it/s]


  Generating 569 synthetic samples...
  [OK] CTGAN completed in 9.66s
  Synthetic data shape: (569, 6)

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Training TVAE...
  Generating 569 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 4.91s
  Synthetic data shape: (569, 6)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Device: cuda
  Training CopulaGAN...
  Generating 569 synthetic samples...
  [OK] CopulaGAN completed in 6.10s
  Synthetic data shape: (569, 6)

[4/7] Training CTABGAN...
--------------------------------------------------
  Device: cuda
  Training CTABGAN...


100%|██████████| 300/300 [00:13<00:00, 21.44it/s]


Finished training in 14.647144794464111  seconds.
  Generating 569 synthetic samples...
  [OK] CTABGAN completed in 14.68s
  Synthetic data shape: (569, 6)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Training CTABGAN+...


100%|██████████| 400/400 [00:13<00:00, 29.84it/s]


Finished training in 14.055717945098877  seconds.
  Generating 569 synthetic samples...
  [OK] CTABGAN+ completed in 14.09s
  Synthetic data shape: (569, 6)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Training PATE-GAN...
  Generating 569 synthetic samples...
  [OK] PATE-GAN completed in 2.09s
  Synthetic data shape: (569, 6)

[7/7] Training MEDGAN...
--------------------------------------------------
  Device: cuda
  Training MEDGAN...
  Generating 569 synthetic samples...
  [OK] MEDGAN completed in 6.00s
  Synthetic data shape: (569, 6)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 9.66s
  - TVAE: 4.91s
  - CopulaGAN: 6.10s
  - CTABGAN: 14.68s
  - CTABGAN+: 14.09s
  - PATE-GAN: 2.09s
  - MEDGAN: 6.00s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan', 'synthet

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

if checkpoint.exists('section_3.2'):
    section3_results = checkpoint.load('section_3.2')['results']
    print("[RESUME] Loaded Section 3.2 evaluation from checkpoint")
else:
    section3_results = evaluate_all_available_models(
        section_number=3,
        scope=globals(),
        models_to_evaluate=None,
        real_data=None,
        target_col=None,
        protected_col=NOTEBOOK_CONFIG.get("protected_col")
    )
    checkpoint.save('section_3.2', {'results': section3_results})

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: breast-cancer-data
[INFO] Target column: diagnosis
[INFO] Protected column: None (fairness metrics skipped)
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/breast-cancer-data/2026-03-26/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.684

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:

- **Smoke mode** (`tuning_mode="smoke"`): Runs 10 pilot trials per model, then displays
  time-budget recommendations (how many trials fit in 1 hour, how long 20 trials would take).
  Section 5 uses the pilot results directly.
- **Full mode** (`tuning_mode="full"`): Runs a pilot phase, displays time estimates, then
  presents three continuation strategies in cell 4.3.  The user reviews the estimates and
  **uncomments one option** before running the cell.

### 4.1 Configuration

Create the `StagedOptimizationManager`.  `TUNING_MODE` and `PILOT_TRIALS` are derived
from `NOTEBOOK_CONFIG` so the same code works for both smoke and full runs.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig,
    flush_previous_runs
)

# Flush optimization studies if FRESH_START is set
if FRESH_START:
    flush_previous_runs(DATASET_IDENTIFIER)

# Derive tuning mode and pilot size from NOTEBOOK_CONFIG
TUNING_MODE = NOTEBOOK_CONFIG.get("tuning_mode", "smoke")
PILOT_TRIALS = 10 if TUNING_MODE == "smoke" else NOTEBOOK_CONFIG.get("n_trials_smoke", 10)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=PILOT_TRIALS,
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Tuning mode: {TUNING_MODE}")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")
print(f"  FRESH_START: {FRESH_START}")

[FLUSH] No previous studies found in results/breast-cancer-data/optimization_studies — starting clean
Staged Optimization Manager created!
  Tuning mode: full
  Pilot trials: 15
  Run mode: full
  Persistence dir: results/breast-cancer-data/optimization_studies
  FRESH_START: True


### 4.2 Run Pilot Phase

Run a pilot phase to establish time estimates.

- **Smoke mode**: 10 trials + smoke recommendations table (trials in 1 hr, time for 20 trials).
- **Full mode**: Pilot trials + time estimates for planning continuation.

In [8]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE
# ============================================================================

# Run pilot phase (uses PILOT_TRIALS from cell 4.1)
pilot_states = manager.run_pilot(
    models_to_run=models_to_run
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

# Smoke mode: show budget recommendations
if TUNING_MODE == "smoke":
    print("\n" + "="*60)
    print("SMOKE MODE - TIME BUDGET RECOMMENDATIONS")
    print("="*60)
    smoke_recs = manager.get_smoke_recommendations()
    print(smoke_recs.to_string(index=False))
    print("\nTo run a full optimization, set tuning_mode='full' in NOTEBOOK_CONFIG")
    print("and use the recommended_pilot column to guide n_trials_smoke.")

[I 2026-03-26 17:50:56,360] A new study created in memory with name: ctgan_hpo_breast-cancer-data



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 15
Dataset identifier: breast-cancer-data


[PILOT] Optimizing CTGAN...
--------------------------------------------------


Gen. (-0.85) | Discrim. (0.03): 100%|██████████| 100/100 [00:02<00:00, 37.46it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3458
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5501
[CHART] Combined Score: 0.4275 (Similarity: 0.3458, Accuracy: 0.5501)
[ctgan] Trial 1: Combined Score: 0.4275 (Similarity: 0.3458, Accuracy: 0.5501) Best Score so far: 0.4275
[ctgan] Trial 2: Combined Score: 0.0000 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.4275


Gen. (-0.31) | Discrim. (-0.13): 100%|██████████| 850/850 [00:22<00:00, 37.67it/s]


...

Finished training in 33.54915428161621  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5905
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8814
[CHART] Combined Score: 0.7069 (Similarity: 0.5905, Accuracy: 0.8814)
[ctabganplus] Trial 13: Combined Score: 0.7069 (Similarity: 0.5905, Accuracy: 0.8814) Best Score so far: 0.7278


100%|██████████| 450/450 [00:21<00:00, 21.33it/s]


Finished training in 21.756625652313232  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5889
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8946
[CHART] Combined Score: 0.7111 (Similarity: 0.5889, Accuracy: 0.8946)
[ctabganplus] Trial 14: Combined Score: 0.7111 (Similarity: 0.5889, Accuracy: 0.8946) Best Score so far: 0.7278


100%|██████████| 750/750 [00:35<00:00, 21.30it/s]


Finished training in 35.86930871009827  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.6053
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8981
[CHART] Combined Score: 0.7224 (Similarity: 0.6053, Accuracy: 0.8981)
[ctabganplus] Trial 15: Combined Score: 0.7224 (Similarity: 0.6053, Accuracy: 0.8981) Best Score so far: 0.7278
  [OK] CTABGAN+: 15 trials in 599.6s
  Best score: 0.7278

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.3051
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2794
[CHART] Combined Score: 0.2948 (Similarity: 0.3051, Accuracy: 0.2794)
[pategan] Trial 1: Combined Score: 0.2948 (Similarity: 0.3051, Accuracy: 0.2794) Best Score so far: 0.2948
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity A

In [9]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      15    0.645928          0.088915           0.218386          False             Continue - still improving
       tvae      15    0.708490          0.000000           0.026165           True                 Stop - plateau reached
  copulagan      15    0.633871          0.000000           0.231713           True                 Stop - plateau reached
    ctabgan      15    0.718871          0.000000           0.001777           True                 Stop - plateau reached
ctabganplus      15    0.727769          0.012215           0.014130          False Consider stopping - minor improvements
    pategan      15    0.447573          0.000000           0.152728           True                 Stop - plateau reached
     medgan      15    0.349361          0.000000           0.098557           True                 Stop - pla

### 4.3 Continuation (Full Mode Only)

**Full mode only** - Review the pilot results and time estimates above, then
uncomment **one** of the three options below and adjust the values before running.

- **Option (i)**: Common trial count for all models
- **Option (ii)**: Per-model trial counts
- **Option (iii)**: Time-based budget (minutes per model)

Models not in `models_to_run` are silently ignored, so listing all 8 is safe.

In [10]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Full mode only - choose ONE option)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping continuation - using pilot results for Section 5.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # continuation_states = manager.continue_optimization(additional_trials=30)

    # OPTION (ii): Per-model trials - adjust counts per model
    continuation_states = manager.continue_optimization(
        trials_per_model={
            'ctgan': 10,
            'tvae':10,
            'copulagan': 10,
            'ctabgan': 10,
            'ctabganplus': 10,
            'ganeraid': 10,
            'pategan': 10,
            'medgan': 10,
        }
    )

    # OPTION (iii): Time-based budget - minutes per model
    # continuation_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         'ctgan': 60,
    #         'tvae': 60,
    #         'copulagan': 60,
    #         'ctabgan': 60,
    #         'ctabganplus': 60,
    #         'ganeraid': 60,
    #         'pategan': 60,
    #         'medgan': 60,
    #     }
    # )

    print("[FULL MODE] Uncomment one option above and re-run this cell.")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 10 additional trials
  tvae: 10 additional trials
  copulagan: 10 additional trials
  ctabgan: 10 additional trials
  ctabganplus: 10 additional trials
  ganeraid: 10 additional trials
  pategan: 10 additional trials
  medgan: 10 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 15 existing trials


Gen. (-0.34) | Discrim. (-0.13): 100%|██████████| 700/700 [00:18<00:00, 36.95it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4655
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6599
[CHART] Combined Score: 0.5432 (Similarity: 0.4655, Accuracy: 0.6599)
[ctgan] Trial 16: Combined Score: 0.5432 (Similarity: 0.4655, Accuracy: 0.6599) Best Score so far: 0.6459


Gen. (-0.40) | Discrim. (-0.30): 100%|██████████| 850/850 [00:23<00:00, 36.78it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4295
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7786
[CHART] Combined Score: 0.5691 (Similarity: 0.4295, Accuracy: 0.7786)
[ctgan] Trial 17: Combined Score: 0.5691 (Similarity: 0.4295, Accuracy: 0.7786) Best Score so far: 0.6459
[ctgan] Trial 18: Combined Score: 0.0000 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.6459


...

Finished training in 69.51665496826172  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.6179
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8998
[CHART] Combined Score: 0.7307 (Similarity: 0.6179, Accuracy: 0.8998)
[ctabganplus] Trial 23: Combined Score: 0.7307 (Similarity: 0.6179, Accuracy: 0.8998) Best Score so far: 0.7307


100%|██████████| 850/850 [01:08<00:00, 12.34it/s]


Finished training in 69.54800462722778  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.6450
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8893
[CHART] Combined Score: 0.7427 (Similarity: 0.6450, Accuracy: 0.8893)
[ctabganplus] Trial 24: Combined Score: 0.7427 (Similarity: 0.6450, Accuracy: 0.8893) Best Score so far: 0.7427


100%|██████████| 1000/1000 [01:21<00:00, 12.28it/s]


Finished training in 82.12142491340637  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.6486
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9069
[CHART] Combined Score: 0.7519 (Similarity: 0.6486, Accuracy: 0.9069)
[ctabganplus] Trial 25: Combined Score: 0.7519 (Similarity: 0.6486, Accuracy: 0.9069) Best Score so far: 0.7519
  [OK] CTABGAN+: 10 trials in 550.8s
  Best score: 0.7519

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 15 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2880
[PRUNED] Trial pruned after similarity calculation (score: 0.2880)
[pategan] Trial 16: Combined Score: 0.2880 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.4476
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 va

In [11]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS (post-continuation)
# ============================================================================

if TUNING_MODE == "full":
    print("DIMINISHING RETURNS ANALYSIS")
    print("="*60)

    convergence_report = manager.get_diminishing_returns_report()

    if len(convergence_report) > 0:
        print(convergence_report.to_string(index=False))

        print("\nInterpretation:")
        print("-" * 40)
        for _, row in convergence_report.iterrows():
            print(f"  {row['model']}: {row['recommendation']}")
            if row['has_plateaued']:
                print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
            else:
                print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
    else:
        print("No convergence data available")
else:
    print("[SMOKE MODE] Skipping post-continuation analysis.")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      25    0.645928          0.000000           0.218386           True                 Stop - plateau reached
       tvae      25    0.713579          0.007132           0.031254           True                 Stop - plateau reached
  copulagan      25    0.661218          0.000000           0.259060           True                 Stop - plateau reached
    ctabgan      25    0.737631          0.000000           0.020538           True                 Stop - plateau reached
ctabganplus      25    0.751893          0.032085           0.038254          False Consider stopping - minor improvements
    pategan      25    0.447573          0.000000           0.152728           True                 Stop - plateau reached
     medgan      25    0.482977          0.000000           0.232174           True                 Stop - pla

### 4.5 Additional Batches (Full Mode, Optional)

If the diminishing returns analysis suggests continuing, uncomment one option below.
You can re-run this cell multiple times.

In [12]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL
# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Full mode only - optional, can repeat)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping additional batches.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # additional_states = manager.continue_optimization(additional_trials=20)

    # OPTION (ii): Per-model trials - adjust counts per model
    additional_states = manager.continue_optimization(
          trials_per_model={
              'ctgan': 1,
              'tvae': 1,
              'copulagan': 1,
              'ctabgan': 1,
              'ctabganplus': 1,
              'ganeraid': 1,
              'pategan': 1,
              'medgan': 1,
          }
      )

    # OPTION (iii): Time-based budget - minutes per model
    # additional_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         'ctgan': 30,
    #         'tvae': 30,
    #         'copulagan': 30,
    #         'ctabgan': 30,
    #         'ctabganplus': 30,
    #         'ganeraid': 30,
    #         'pategan': 30,
    #         'medgan': 30,
    #     }
    # )

    # print("\nUpdated convergence report:")
    # print(manager.get_diminishing_returns_report().to_string(index=False))

    print("Cell skipped - uncomment an option above to run additional batches")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 1 additional trials
  tvae: 1 additional trials
  copulagan: 1 additional trials
  ctabgan: 1 additional trials
  ctabganplus: 1 additional trials
  ganeraid: 1 additional trials
  pategan: 1 additional trials
  medgan: 1 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 25 existing trials


Gen. (-0.16) | Discrim. (-0.42): 100%|██████████| 800/800 [00:21<00:00, 37.55it/s]


[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4836
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7074
[CHART] Combined Score: 0.5731 (Similarity: 0.4836, Accuracy: 0.7074)
[ctgan] Trial 26: Combined Score: 0.5731 (Similarity: 0.4836, Accuracy: 0.7074) Best Score so far: 0.6459
  [OK] CTGAN: 1 trials in 25.4s
  Best score: 0.6459

[CONTINUE] Optimizing TVAE...
--------------------------------------------------
  Resuming from 25 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5141
[PRUNED] Trial pruned after similarity calculation (score: 0.5141)
[tvae] Trial 26: Combined Score: 0.5141 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.7136
  [OK] TVAE: 1 trials in 8.2s
  Best score: 0.7136

[CONTINUE] Optimizing CopulaGAN...
--------------------------------------------------
  Resuming from 25 existing tri

100%|██████████| 750/750 [00:34<00:00, 21.65it/s]


Finished training in 35.296396255493164  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5801
[PRUNED] Trial pruned after similarity calculation (score: 0.5801)
[ctabgan] Trial 26: Combined Score: 0.5801 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.7376
  [OK] CTABGAN: 1 trials in 35.4s
  Best score: 0.7376

[CONTINUE] Optimizing CTABGAN+...
--------------------------------------------------
  Resuming from 25 existing trials


100%|██████████| 1000/1000 [01:21<00:00, 12.32it/s]


Finished training in 81.8409914970398  seconds.
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5479
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9112
[CHART] Combined Score: 0.6932 (Similarity: 0.5479, Accuracy: 0.9112)
[ctabganplus] Trial 26: Combined Score: 0.6932 (Similarity: 0.5479, Accuracy: 0.9112) Best Score so far: 0.7519
  [OK] CTABGAN+: 1 trials in 82.1s
  Best score: 0.7519

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 25 existing trials
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2839
[PRUNED] Trial pruned after similarity calculation (score: 0.2839)
[pategan] Trial 26: Combined Score: 0.2839 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.4476
  [OK] PATE-GAN: 1 trials in 1.9s
  Best score: 0.4476

[CONTINUE] Optimizing MEDGAN...
-----------------

### 4.6 Save Best Parameters

In [13]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/breast-cancer-data/2026-03-26/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.6459

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7376

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7519

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.6612

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.7136

[CHART] Processing PATE-GAN parameters...
[OK] Found PATE-GAN: 10 parameters, score: 0.4476

[CHART] Processing MEDGAN parameters...
[OK] Found MEDGAN: 11 parameters, score: 0.4830

[OK] Parameters saved: results/breast-cancer-data/2026-03-26/Section-4/best_parameters.csv
   - Total parameter entries: 67
[OK] Summary sav

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [14]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4 (checkpoint resumes completed models)
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True,
    checkpoint=checkpoint
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (569, 6)
Target column: diagnosis
Samples to generate: 569
Loading parameters from: Section 4
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/breast-cancer-data/2026-03-26/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters
[OK] Loaded PATE-GAN: 10 parameters
[OK] Loaded MEDGAN: 11 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 7
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - copulaga

Gen. (-0.29) | Discrim. (-0.16): 100%|██████████| 1000/1000 [00:14<00:00, 70.41it/s]


  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.4628
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8163
[CHART] Combined Score: 0.6042 (Similarity: 0.4628, Accuracy: 0.8163)
  [OK] CTGAN completed in 14.96s
  Synthetic data shape: (569, 6)
  Objective score: 0.6042

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 8
    - epochs: 750
    - batch_size: 256
    - learning_rate: 0.0011
    - embedding_dim: 96
    - l2scale: 0.0050
    ... and 4 more
  Training TVAE...
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5287
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9007
[CHART] Combined Score: 0.6775 (Similarity: 0.5287, Accuracy: 0.9007)
  [OK] TVAE completed in 11.74s
  Synthetic data shape: (569, 6

100%|██████████| 800/800 [00:36<00:00, 21.74it/s]


Finished training in 37.462876319885254  seconds.
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5800
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8989
[CHART] Combined Score: 0.7076 (Similarity: 0.5800, Accuracy: 0.8989)
  [OK] CTABGAN completed in 37.50s
  Synthetic data shape: (569, 6)
  Objective score: 0.7076

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 2
    - epochs: 1000
    - batch_size: 128
    - categorical_columns: []
    - target_col: diagnosis
  Training CTABGAN+...


100%|██████████| 1000/1000 [01:20<00:00, 12.35it/s]


Finished training in 81.62871956825256  seconds.
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.5538
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8972
[CHART] Combined Score: 0.6912 (Similarity: 0.5538, Accuracy: 0.8972)
  [OK] CTABGAN+ completed in 81.66s
  Synthetic data shape: (569, 6)
  Objective score: 0.6912

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 10
    - epochs: 500
    - batch_size: 32
    - num_teachers: 20
    - generator_lr: 0.0009
    - discriminator_lr: 0.0000
    ... and 6 more
  Training PATE-GAN...
  Generating 569 synthetic samples...
[TARGET] Enhanced objective function using target column: 'diagnosis'
[OK] Similarity Analysis: 6/6 valid metrics, Average: 0.2917
[ERROR] TRTS (Real->Synthetic) failed: Classification metrics can't handle a mix of continuous and binary targets
[ER

### 5.2 Batch Evaluation of Optimized Models

In [15]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

if checkpoint.exists('section_5.2'):
    section5_batch_results = checkpoint.load('section_5.2')['results']
    print("[RESUME] Loaded Section 5.2 evaluation from checkpoint")
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
else:
    try:
        section5_batch_results = evaluate_section5_optimized_models(
            section_number=5,
            scope=globals(),
            target_column=TARGET_COLUMN,
            protected_col=NOTEBOOK_CONFIG.get("protected_col"),
            compute_mia=True
        )
        checkpoint.save('section_5.2', {'results': section5_batch_results})
        
        print("\n" + "="*80)
        print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
        print("="*80)
        print(f"Models processed: {section5_batch_results['models_processed']}")
        print(f"Results exported to: {section5_batch_results['results_dir']}")
        
    except Exception as e:
        print(f"Section 5.2 batch evaluation failed: {e}")
        import traceback
        traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: breast-cancer-data
[INFO] Target column: diagnosis
[INFO] Protected column: None (fairness metrics skipped)
[INFO] MIA evaluation: ON
[INFO] Variable pattern: final
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/breast-cancer-data/2026-03-26/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.798

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.026
   [CHART] Explained Variance (PC1, PC2): 0.638, 0.172

[3] D

### 5.3 Final Summary

In [16]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: breast-cancer-data
Session: 2026-03-26

Results directories:
  Section 2 (EDA): results/breast-cancer-data/2026-03-26/Section-2
  Section 3 (Demo): results/breast-cancer-data/2026-03-26/Section-3
  Section 4 (HPO): results/breast-cancer-data/2026-03-26/Section-4
  Section 5 (Final): results/breast-cancer-data/2026-03-26/Section-5

Staged Optimization Summary:
------------------------------------------------------------
      model  best_score  total_trials    status
      ctgan    0.645928            26 completed
       tvae    0.713579            26 completed
  copulagan    0.661218            26 completed
    ctabgan    0.737631            26 completed
ctabganplus    0.751893            26 completed
    pategan    0.447573            26 completed
     medgan    0.482977            26 completed

Final Model Performance (with optimized parameters):
------------------------------------------------------------
  1. CTABGAN: score=0.7076